# EndurancePy — full quickstart

A guided tour of the package on real endurance data (Al Kamel archives).
**Requires network access.** Inspired by FastF1.

> No Al Kamel data is bundled with the project — please respect the portals' terms of service.

In [ ]:
import endurancepy as ep

ep.Cache.enable_cache("./cache")  # the directory must exist

## 1. Season calendar

Season ids look like `08_2018-2019` / `13_2024` (see the portal URL).

In [ ]:
schedule = ep.get_event_schedule(2019, "WEC", season="08_2018-2019")
schedule[["RoundNumber", "EventName", "EventDate", "Sessions"]]

## 2. Load a session

Pick an event from the schedule, then load its race (the session already knows its season).

In [ ]:
event = schedule.get_event_by_name("Spa")
session = event.get_race()
session.load()
len(session.laps), len(session.cars)

## 3. Classification (overall & per class)

In [ ]:
session.results[["Position", "CarNumber", "Class", "Crew", "Laps", "BestLapTime"]].head(
    10
)

In [ ]:
session.results.pick_classes("LMGTE Am").head()

## 4. Laps and the `pick_*` filters

In [ ]:
laps = session.laps

for class_name in sorted(laps["Class"].dropna().unique()):
    best = laps.pick_classes(class_name).pick_fastest()
    if best is not None:
        print(f"{class_name:10} {best['LapTime']}  (car {best['CarNumber']})")

In [ ]:
# Clean green-flag laps of the leading car, by stint
lead = session.cars[0]
car = laps.pick_cars(lead).pick_wo_box().pick_track_status("GF")
car.groupby("Stint")["LapTime"].agg(["count", "min"])

## 5. Track status & weather

In [ ]:
session.track_status

In [ ]:
session.weather_data.head()

## 6. Plotting (colours by class)

In [ ]:
import matplotlib.pyplot as plt

from endurancepy import plotting

plotting.setup_mpl()
clean = laps.pick_wo_box().pick_track_status("GF")

fig, ax = plt.subplots()
for class_name in sorted(laps["Class"].dropna().unique()):
    for _, cl in clean.pick_classes(class_name).groupby("CarNumber"):
        ax.plot(
            cl["LapNumber"],
            cl["LapTime"].dt.total_seconds(),
            ".",
            color=plotting.get_class_color(class_name),
            alpha=0.5,
        )
ax.set_xlabel("Lap")
ax.set_ylabel("Lap time (s)")
ax.set_title("Pace by class")
plt.show()

## 7. Championship standings

Pass one `session.results` per round to `compute_standings` (configurable points / per class).

In [ ]:
# Illustrative: a single round here; in practice collect every round's results.
ep.compute_standings([session.results], by="CarNumber", per_class=True).head(10)